# Chaos Index Tutorial

This notebook computes a frame-level **Chaos Index (0-100)** from SkillCorner tracking data and provides:

- a timeline of chaos over the match,
- the top chaotic moments,
- overlay-ready chaotic segments for video annotation.

The implementation uses `src/features/ChaosIndex.py`.

In [8]:
import json
import sys
from pathlib import Path

import pandas as pd

# Make `src` importable even when notebook kernel starts in the notebook folder.
if (Path.cwd() / "src").exists():
    repo_root = Path.cwd()
elif (Path.cwd().parent / "src").exists():
    repo_root = Path.cwd().parent
elif (Path.cwd().parent.parent / "src").exists():
    repo_root = Path.cwd().parent.parent
else:
    # Fallback for this tutorial path: notebooks/tutorials/03_Basics_of_Tracking
    repo_root = Path.cwd().parent.parent.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.features.ChaosIndex import ChaosIndexCalculator

## 1) Load match tracking data

Set your `match_id` and load tracking + match metadata.

In [9]:
match_id = 1886347

tracking_path = repo_root / "data" / "matches" / str(match_id) / f"{match_id}_tracking_extrapolated.jsonl"
match_path = repo_root / "data" / "matches" / str(match_id) / f"{match_id}_match.json"

with open(tracking_path, "rb") as f:
    header = f.read(128)

if b"git-lfs.github.com/spec/v1" in header:
    raise RuntimeError(
        "Tracking file is an LFS pointer, not real frame data.\n\n"
        "Your current folder is not a git clone and git-lfs is not available, so `git lfs pull` cannot run here.\n\n"
        "Fix options:\n"
        "1) Re-clone with Git LFS enabled:\n"
        "   - brew install git-lfs\n"
        "   - git lfs install\n"
        "   - git clone https://github.com/SkillCorner/opendata.git\n"
        "2) Replace this tracking file with the real JSONL from an LFS-enabled clone.\n\n"
        "Then rerun this cell."
    )

tracking_df = pd.read_json(tracking_path, lines=True)
with open(match_path, "r") as f:
    match_metadata = json.load(f)

print(f"Loaded {len(tracking_df):,} frames for match {match_id}")

RuntimeError: Tracking file is a Git LFS pointer. Pull LFS data first (git lfs pull), then rerun.

## 2) Compute frame-level Chaos Index

In [ ]:
chaos_calc = ChaosIndexCalculator(
    fps=10.0,
    rolling_window_frames=300,
    chaos_quantile_for_flags=0.95,
)

chaos_df = chaos_calc.calculate(tracking_df=tracking_df, match_metadata=match_metadata)
chaos_df.head()

## 3) Timeline + top chaotic moments

In [ ]:
top_moments = chaos_calc.top_chaotic_moments(
    chaos_df,
    top_n=10,
    min_separation_frames=50,
)

display(top_moments[["frame", "timestamp", "chaos_index"]])

fig, ax = chaos_calc.plot_timeline(
    chaos_df,
    top_n=10,
    min_separation_frames=50,
)

## 4) Build overlay segments for video

These segments can be used to add labels like *"chaotic phase"* on video.

In [ ]:
overlay_segments = chaos_calc.overlay_segments(
    chaos_df,
    min_segment_len_frames=20,
)

display(overlay_segments.head(20))

## 5) Export outputs

In [ ]:
out_dir = repo_root / "data" / "matches" / str(match_id)
out_dir.mkdir(parents=True, exist_ok=True)

chaos_timeline_path = out_dir / f"{match_id}_chaos_timeline.csv"
top_moments_path = out_dir / f"{match_id}_chaos_top_moments.csv"
overlay_segments_path = out_dir / f"{match_id}_chaos_overlay_segments.csv"

chaos_df.to_csv(chaos_timeline_path, index=False)
top_moments.to_csv(top_moments_path, index=False)
overlay_segments.to_csv(overlay_segments_path, index=False)

print("Saved:")
print("-", chaos_timeline_path)
print("-", top_moments_path)
print("-", overlay_segments_path)